# ONS - Notícias (produção)

Notebook próprio (não encaixa nos 3 dispatchers genéricos — a fonte não tem
HTML pra raspar, é consumida via API REST do proxy do site). Ver
`ingestores/ENERGIA/teste_ons_noticias.ipynb` para o notebook de teste da
Fase 1 que descobriu os endpoints.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"

In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

SOURCE_ID = "ons_noticias"
SOURCE_DESCRICAO = "Linked from ONS — Notícias"
SITE_URL = "https://www.ons.org.br/paginas/imprensa/noticias"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}/ENERGIA"
os.makedirs(PASTA_DESTINO, exist_ok=True)

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)
CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

# Proxy REST que o próprio site consome via fetch() — descoberto lendo
# wp_noticias.js / wp_noticiasDetalhe.js. Sem HTML pra raspar aqui: a
# listagem e o detalhe da notícia vêm inteiros de duas chamadas de API.
PROXY_BASE = "https://proxyportais.ons.org.br/ons.portalempregado.proxy/"
LISTA_NOTICIAS = "Multicanais - Notícias"
URL_LISTAGEM = (
    PROXY_BASE
    + "attachment/_api/web/lists/getbytitle('"
    + urllib.parse.quote(LISTA_NOTICIAS, safe="")
    + "')/items"
)
URL_DETALHE = PROXY_BASE + "api/noticias/get"
URL_PAGINA_NOTICIA = "https://www.ons.org.br/paginas/noticias/details.aspx?i="
TAXONOMIA_SITE_ONS = "902c3577-c4b1-4d76-9780-bcb001c0e0c3"

# Janela de captura por execução: paginado por $skiptoken, então limitar aqui
# evita percorrer o histórico inteiro a cada rodada do job diário. 30 itens
# (3 páginas de 10) cobre folgado o volume observado (2-3 notícias/dia).
MAX_ITENS_POR_EXECUCAO = 30
ITENS_POR_PAGINA = 10
MIN_CHARS_TEXTO = 200

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

# Origin/Referer não são decorativos aqui: sem eles o proxy responde 405.
HEADERS_API = {
    "User-Agent": USER_AGENT,
    "Accept": "application/json;odata=verbose",
    "Content-Type": "application/json;odata=verbose",
    "Odata-Version": "3.0",
    "CacheLevel": "User",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Origin": "https://www.ons.org.br",
    "Referer": SITE_URL,
}

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form", "button"]

In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception as e:
        print(f"[manifesto] falha ao carregar ({e}); iniciando vazio.")
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


def baixar_json(url: str, params: Optional[dict] = None, tentativas: int = 3) -> Optional[dict]:
    """Baixa um JSON do proxy do ONS, com httpx e fallback em curl_cffi."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = httpx.get(url, params=params, headers=HEADERS_API,
                             timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text:
                return resp.json()
            print(f"  [httpx tent {tentativa}/{tentativas}] HTTP {resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, params=params, headers=HEADERS_API,
                                     impersonate=impersonate, timeout=HTTP_TIMEOUT,
                                     allow_redirects=True)
            if resp.status_code == 200 and resp.text:
                return resp.json()
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] HTTP {resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

In [0]:
# =============================================================================
# Etapa 1 — Listar notícias novas
# =============================================================================

def _filtro_publicadas() -> str:
    """Filtro OData equivalente ao que o site monta em m_setFilters() (wp_noticias.js)."""
    agora = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    return (
        f"PublishStatus eq 'Aprovado' and "
        f"(PublishDate le datetime'{agora}' and "
        f"(ExpirationDate eq null or ExpirationDate ge datetime'{agora}')) and "
        f"TaxCatchAll/IdForTerm eq '{TAXONOMIA_SITE_ONS}'"
    )


def _extrair_skiptoken(url_next: Optional[str]) -> Optional[str]:
    """Pega só o $skiptoken do __next — o host que vem nele é interno, não
    acessível de fora (admmulticanais.ons.org.br)."""
    if not url_next:
        return None
    query = urllib.parse.urlparse(url_next).query
    valores = urllib.parse.parse_qs(query).get("$skiptoken")
    return valores[0] if valores else None


def listar_noticias(max_itens: int, por_pagina: int) -> Optional[list[dict]]:
    """Retorna None se a primeira página falhar (fonte indisponível);
    lista vazia/parcial se falhar numa página seguinte (mantém o que já
    coletou em vez de descartar tudo)."""
    itens, skiptoken = [], None

    while len(itens) < max_itens:
        params = {
            "$select": "Id,Title,Subtitle,PublishDate",
            "$filter": _filtro_publicadas(),
            "$orderby": "PublishDate desc,Title asc",
            "$top": str(por_pagina),
        }
        if skiptoken:
            params["$skiptoken"] = skiptoken

        dados = baixar_json(URL_LISTAGEM, params=params)
        if not dados:
            if not itens:
                return None
            print("  -> falha ao baixar página seguinte da listagem; seguindo com o que já tem.")
            break

        bloco = (dados.get("d") or {}).get("results") or []
        if not bloco:
            break

        for registro in bloco:
            publicado = registro.get("PublishDate")
            itens.append({
                "id": registro.get("Id"),
                "titulo": registro.get("Title"),
                "subtitulo": registro.get("Subtitle"),
                "published_at": publicado[:10] if publicado else None,
                "url": f"{URL_PAGINA_NOTICIA}{registro.get('Id')}",
            })

        skiptoken = _extrair_skiptoken((dados.get("d") or {}).get("__next"))
        if not skiptoken:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens[:max_itens]

In [0]:
# =============================================================================
# Etapa 2 — Abrir notícia individual e extrair texto completo
# =============================================================================

def extrair_noticia(id_noticia) -> Optional[dict]:
    dados = baixar_json(URL_DETALHE, params={"id": str(id_noticia)})
    if not dados:
        return None

    html_conteudo = dados.get("ContentSimple") or ""
    try:
        soup = BeautifulSoup(html_conteudo, "lxml")
    except Exception:
        soup = BeautifulSoup(html_conteudo, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    texto = PADRAO_LINHAS_VAZIAS.sub("\n\n", soup.get_text("\n", strip=True)).strip()

    publicado = dados.get("PublishDate")
    return {
        "id": dados.get("ID") or id_noticia,
        "titulo": dados.get("Title"),
        "subtitulo": dados.get("Subtitle"),
        "published_at": publicado[:10] if publicado else None,
        "url": f"{URL_PAGINA_NOTICIA}{id_noticia}",
        "texto": texto,
    }

In [0]:
# =============================================================================
# Etapa 3 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json

In [0]:
# =============================================================================
# Execução
# =============================================================================

try:
    ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)

    noticias = listar_noticias(MAX_ITENS_POR_EXECUCAO, ITENS_POR_PAGINA)

    if noticias is None:
        print("=== falha ao baixar a listagem de notícias. ===")
        atualizar_status_fonte(
            source_id=SOURCE_ID, sucesso=False, docs_capturados=0,
            erro="download da listagem falhou",
        )
    else:
        itens_novos = [n for n in noticias if n["url"] not in ja_processados]
        print(f"{len(noticias)} itens na listagem, {len(itens_novos)} novos.")

        salvos = 0
        for item in itens_novos:
            print(f"\n  [item] {item['titulo'][:100]}")
            detalhe = extrair_noticia(item["id"])

            if detalhe is None:
                print("    -> download do detalhe falhou; pulando.")
                continue

            if not detalhe["texto"] or len(detalhe["texto"]) < MIN_CHARS_TEXTO:
                print(f"    -> texto muito curto ({len(detalhe['texto'])} chars); pulando.")
                continue

            metadados = {
                "source_id": SOURCE_ID,
                "title": detalhe["titulo"] or item["titulo"],
                "description": SOURCE_DESCRICAO,
                "url": detalhe["url"],
                "date": HOJE,
                "published_at": detalhe["published_at"] or item["published_at"],
            }

            caminho_txt, _ = salvar_artefatos(PASTA_DESTINO, metadados["title"], detalhe["texto"], metadados)
            print(f"    -> salvo em {caminho_txt}")

            ja_processados.add(item["url"])
            salvos += 1
            time.sleep(random.uniform(0.5, 1.2))

        salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)
        print(f"\n=== {salvos} notícia(s) nova(s) salva(s). ===")
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=salvos)

except Exception as e:
    print(f"=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=str(e))

print("\n=== Fim. ===")